# Validasi Grid — Orang A (2026)

Grid yang sudah dibuat Arya **tidak ditimpa**. Notebook ini melakukan pemeriksaan ID, koordinat, geometri, dan sebaran enam provinsi.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data' / 'interim' / 'grid.csv').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'interim' / 'grid.csv').exists(), 'Buka notebook dari folder karhutla'
RAW = ROOT / 'data' / 'raw'
INTERIM = ROOT / 'data' / 'interim'
OUTPUT = ROOT / 'data' / 'processing'
grid = pd.read_csv(INTERIM / 'grid.csv')
firms = pd.read_parquet(INTERIM / 'hotspots_daily.parquet')
firms['date'] = pd.to_datetime(firms['date'])
firms = firms.loc[firms.date.dt.year.eq(2026)].copy()
print('Grid total:', len(grid), '| FIRMS agregat 2026:', len(firms))
print('Provinsi:', ', '.join(sorted(grid.province.unique())))


## Integritas grid

In [ ]:
assert grid.cell_id.notna().all()
assert not grid.cell_id.duplicated().any()
assert grid[['lon','lat','province']].notna().all().all()
assert grid.lon.between(95,125).all() and grid.lat.between(-8,10).all()
print('✅ Semua grid memiliki ID unik, provinsi dan koordinat.')
display(grid.groupby('province').agg(n_grid=('cell_id','size'),lon_min=('lon','min'),lon_max=('lon','max'),lat_min=('lat','min'),lat_max=('lat','max')))


In [ ]:
fig, ax=plt.subplots(figsize=(10,5))
for name, group in grid.groupby('province'):
    ax.scatter(group.lon, group.lat, s=0.7, alpha=0.6, label=name)
ax.set_title('Sebaran pusat grid enam provinsi'); ax.set_xlabel('Bujur'); ax.set_ylabel('Lintang')
ax.legend(markerscale=8, bbox_to_anchor=(1.02, 1), loc='upper left'); plt.tight_layout(); plt.show()


## Validasi sumber GEE + FIRMS

In [ ]:
target = OUTPUT / 'dataset_gabungan_2026.parquet'
if target.exists():
    import pyarrow.dataset as ds
    # Audit dilakukan secara batch agar aman untuk jutaan baris.
    scanner = ds.dataset(target).scanner(columns=['cell_id','date','precip_mm','t2m_c','y_7d'],batch_size=250000)
    total = 0; missing_rain = 0; missing_t = 0; positives = 0
    for batch in scanner.to_batches():
        part = batch.to_pandas(); total += len(part)
        missing_rain += int(part.precip_mm.isna().sum()); missing_t += int(part.t2m_c.isna().sum())
        positives += int(part.y_7d.sum())
    print(dict(rows=total, null_hujan=missing_rain, null_suhu=missing_t, kasus_positif=positives))
else:
    print('Output gabungan belum ada: validasi GEE akan berjalan setelah --merge-2026.')
